# 32. 남은 stuck 케이스 원인 조사

## 이번 노트북에서 할 것
- thioester, catechol(재현안됨), Three-membered_heterocycle stuck 원인 조사
- 고칠 수 있는 건 고치고, 안 되면 limitations.md에 정직하게 기록

## 간략한 정리 (31까지)
- 라이브러리 32개 규칙, 11가지 편집 방식
- 오늘(31번) 활성보존 근사지표(Tanimoto 평균 0.539, 3D형태 BCP사례
  10%이내 변화), SA Score(-0.088, 90% 유지개선), ablation 재분석으로
  버그 2건 발견(재시도로직 부재, replace_ring 조각화) - 수정 후
  5개 사례 중 4개가 규칙기반 단독으로 success 도달
- 외부평가(80/100)+지도교수 피드백 반영 중, 문서 2건 제안서 첨부용 작성
- 향후 확장 논의 중: 다중 전문가 페르소나 에이전트, 라이브러리 자동확장
  보조 에이전트 (7일 여유, 비즈니스 가치/워크플로 통합 방향으로 고민 중)
- test set은 여전히 미사용

## 다음에 해야 할 것
- 확장 방향 학생이 고민 후 결정
- phthalimide, hydroxamic_acid 등 문헌형 규칙(학생)
- test set 최종 검증(학생 승인 시)

In [1]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 378, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 378 (delta 61), reused 90 (delta 38), pack-reused 260 (from 1)
Receiving objects: 100% (378/378), 984.96 KiB | 4.97 MiB/s, done.
Resolving deltas: 100% (198/198), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib, random, json
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, canonicalize, iterative_fix_loop

data = load_tox21_clean(random_state=7)
print(f"도구 로드 완료. 라이브러리 규칙 수: {len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])}")

[06:30:32] WARNING: not removing hydrogen atom without neighbors
[06:30:32] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:30:33] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:30:33] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:30:33] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:30:34] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:30:34] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:30:34] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:30:34] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:30:34] WARNING: not removing hydrogen atom without neighbors


도구 로드 완료. 라이브러리 규칙 수: 33


In [5]:
stuck_targets = {
    "thioester": None,
    "catechol": None,
    "Three-membered_heterocycle": None,
}

for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in stuck_targets and stuck_targets[p['rule_name']] is None:
            result = propose_fix(s, p['rule_name'], candidate_idx=0)
            stuck_targets[p['rule_name']] = {"smiles": s, "atom_indices": p['atom_indices'], "propose_fix_result": result}

for rule, info in stuck_targets.items():
    print(f"=== {rule} ===")
    print(info)
    print()

=== thioester ===
{'smiles': 'O=C(O)CSCC(=O)NC1CCSC1=O', 'atom_indices': [12, 13, 14], 'propose_fix_result': {'new_smiles': 'O=C(O)CSCC(=O)NC1CCOC1=O', 'candidate_used': 'ester (O replacing S)', 'rationale': '티오에스터의 황을 산소로 대체하여 일반 에스터로 전환. 티오에스터는 일반 에스터보다 가수분해 반응성이 높고 아실화 능력이 강해 단백질 등과 부반응 우려가 있음 (검증 필요)', 'is_valid': True}}

=== catechol ===
{'smiles': 'O=C(C=Cc1ccc(O)c(O)c1)O[C@@H]1C[C@@](OC(=O)C=Cc2ccc(O)c(O)c2)(C(=O)O)C[C@@H](O)[C@@H]1O', 'atom_indices': [4, 5, 6, 7, 8, 9, 10, 11], 'propose_fix_result': {'new_smiles': 'COc1ccc(C=CC(=O)O[C@@H]2C[C@@](OC(=O)C=Cc3ccc(O)c(O)c3)(C(=O)O)C[C@@H](O)[C@@H]2O)cc1O', 'candidate_used': 'methoxy', 'rationale': '[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 아니라 약효 상실로 이어짐. 실제 도파민은 도파민 수용체 D1(Ki 4.3-5.6 nM), D2(Ki 4.7-7.2 nM), D3(Ki 6.4-7.3 nM)에 단자릿수 나노몰 수준의 강력한 작용제 친화도를 가짐(IUPHAR/BPS Guide to PHARMACOLOGY 확인). || 인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리

In [6]:
for rule, info in stuck_targets.items():
    smi = info['smiles']
    result = iterative_fix_loop(smi, max_iterations=10)
    print(f"=== {rule} ===")
    print(f"상태: {result['status']}")
    for h in result['history']:
        print(f"  {h}")
    print()

=== thioester ===
상태: success
  {'step': 0, 'smiles': 'O=C(O)CSCC(=O)NC1CCSC1=O', 'problems': [{'rule_name': 'thioester', 'atom_indices': [12, 13, 14]}]}
  {'step': 1, 'smiles': 'O=C(O)CSCC(=O)NC1CCOC1=O', 'fixed_rule': 'thioester', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'ester (O replacing S)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': []}

=== catechol ===
상태: success
  {'step': 0, 'smiles': 'O=C(C=Cc1ccc(O)c(O)c1)O[C@@H]1C[C@@](OC(=O)C=Cc2ccc(O)c(O)c2)(C(=O)O)C[C@@H](O)[C@@H]1O', 'problems': [{'rule_name': 'catechol', 'atom_indices': [4, 5, 6, 7, 8, 9, 10, 11]}, {'rule_name': 'Michael_acceptor_1', 'atom_indices': [0, 1, 2, 3]}]}
  {'step': 1, 'smiles': 'COc1ccc(C=CC(=O)O[C@@H]2C[C@@](OC(=O)C=Cc3ccc(O)c(O)c3)(C(=O)O)C[C@@H](O)[C@@H]2O)cc1O', 'fixed_rule': 'catechol', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'methoxy', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'catechol', 'atom_indices': [19, 20, 21, 22, 23, 24, 25, 26]}, {'ru

In [7]:
!git add -A
!git commit -m "Verify remaining stuck cases (thioester, catechol, Three-membered_heterocycle) are resolved by the retry-logic fix from session 30. All three now reach success or a correctly-classified no_known_fix (Three-membered_heterocycle's epoxide is fixed; remaining cyclooctane_1 is a genuinely uncovered rule, not a bug)."
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [10]:
from rdkit.Chem import rdFingerprintGenerator, Descriptors, Descriptors3D, DataStructs, QED, AllChem

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# SA Score 함수도 이번 노트북에 아직 없으면 다시 로드
import urllib.request, sys
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py",
    "sascorer.py")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz",
    "fpscores.pkl.gz")
sys.path.append('.')
import sascorer

print("준비 완료")

준비 완료


In [11]:
from rdkit.Chem import Descriptors, Descriptors3D, DataStructs, QED
from rdkit.Chem import AllChem

def compute_activity_preservation_metrics(original_smiles, fixed_smiles):
    """치환 전후 분자를 비교해 활성 보존 판단에 필요한 전체 지표 계산."""
    mol_o = Chem.MolFromSmiles(original_smiles)
    mol_f = Chem.MolFromSmiles(fixed_smiles)
    if mol_o is None or mol_f is None:
        return None

    # 2D 화학 연결성
    fp_o = _generator.GetFingerprint(mol_o)
    fp_f = _generator.GetFingerprint(mol_f)
    tanimoto = DataStructs.TanimotoSimilarity(fp_o, fp_f)

    # 약물유사성/물성
    qed_o, qed_f = QED.qed(mol_o), QED.qed(mol_f)
    logp_o, logp_f = Descriptors.MolLogP(mol_o), Descriptors.MolLogP(mol_f)
    sa_o, sa_f = sascorer.calculateScore(mol_o), sascorer.calculateScore(mol_f)

    # 3D 형태 (실패 가능성 있으므로 예외처리)
    def get_3d(mol):
        m = Chem.AddHs(mol)
        if AllChem.EmbedMolecule(m, randomSeed=42) != 0:
            return None
        AllChem.MMFFOptimizeMolecule(m)
        return m

    m3d_o, m3d_f = get_3d(mol_o), get_3d(mol_f)
    shape_available = m3d_o is not None and m3d_f is not None

    result = {
        "tanimoto": tanimoto,
        "delta_qed": qed_f - qed_o,
        "delta_logp": logp_f - logp_o,
        "delta_sa_score": sa_f - sa_o,
        "shape_available": shape_available,
    }

    if shape_available:
        rog_o = Descriptors3D.RadiusOfGyration(m3d_o)
        rog_f = Descriptors3D.RadiusOfGyration(m3d_f)
        result["delta_rog_pct"] = (rog_f - rog_o) / rog_o * 100 if rog_o != 0 else None

    return result


def classify_activity_risk(metrics):
    """지표 조합으로 활성 손실 위험도를 분류."""
    if metrics is None:
        return "판정 불가"

    tanimoto = metrics["tanimoto"]
    qed_ok = abs(metrics["delta_qed"]) < 0.1
    logp_ok = abs(metrics["delta_logp"]) < 1.0
    shape_ok = (metrics.get("delta_rog_pct") is not None
                and abs(metrics["delta_rog_pct"]) < 15) if metrics["shape_available"] else None

    if tanimoto < 0.5:
        if shape_ok and qed_ok and logp_ok:
            return "2D는 크게 변화했으나 형태·물성 보존 (bioisostere 가능성, 활성 유지 가능성 있음)"
        else:
            return "구조·형태·물성 모두 크게 변화 (활성 손실 우려, 검토 권장)"
    else:
        if qed_ok and logp_ok:
            return "국소 치환, 물성 보존 (활성 유지 가능성 높음)"
        else:
            return "구조는 유사하나 물성 변화 큼 (확인 필요)"

In [12]:
test_bcp_orig = "Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1"
test_bcp_fixed = propose_fix(test_bcp_orig, "aniline", candidate_idx=1)['new_smiles']

metrics_bcp = compute_activity_preservation_metrics(test_bcp_orig, test_bcp_fixed)
print("BCP 케이스:", metrics_bcp)
print("판정:", classify_activity_risk(metrics_bcp))

test_anh_orig = "CC(=O)OC(C)=O"
test_anh_fixed = propose_fix(test_anh_orig, "beta-keto/anhydride", candidate_idx=0)['new_smiles']

metrics_anh = compute_activity_preservation_metrics(test_anh_orig, test_anh_fixed)
print("\n무수물 케이스:", metrics_anh)
print("판정:", classify_activity_risk(metrics_anh))

BCP 케이스: {'tanimoto': 0.4888888888888889, 'delta_qed': 0.0014737342178798851, 'delta_logp': -1.0075000000000003, 'delta_sa_score': 1.8787713369816679, 'shape_available': True, 'delta_rog_pct': -7.226480066948701}
판정: 구조·형태·물성 모두 크게 변화 (활성 손실 우려, 검토 권장)

무수물 케이스: {'tanimoto': 0.4166666666666667, 'delta_qed': 0.10841325890450743, 'delta_logp': -0.00510000000000016, 'delta_sa_score': -0.8013209020470491, 'shape_available': True, 'delta_rog_pct': -31.633205860620535}
판정: 구조·형태·물성 모두 크게 변화 (활성 손실 우려, 검토 권장)


In [13]:
def classify_activity_risk_v2(metrics):
    """지표별 세부 판정 + 종합 결론을 함께 반환."""
    if metrics is None:
        return {"verdict": "판정 불가", "details": []}

    details = []

    conn_ok = metrics["tanimoto"] >= 0.5
    details.append(f"2D 연결성: {'유사' if conn_ok else '상이'} (Tanimoto {metrics['tanimoto']:.3f})")

    if metrics["shape_available"] and metrics.get("delta_rog_pct") is not None:
        shape_ok = abs(metrics["delta_rog_pct"]) < 15
        details.append(f"3D 형태: {'보존' if shape_ok else '변화'} (회전반경 {metrics['delta_rog_pct']:+.1f}%)")
    else:
        shape_ok = None
        details.append("3D 형태: 계산 불가")

    qed_ok = abs(metrics["delta_qed"]) < 0.1
    details.append(f"약물유사성(QED): {'유지' if qed_ok else '변화'} ({metrics['delta_qed']:+.3f})")

    logp_ok = abs(metrics["delta_logp"]) < 1.0
    details.append(f"소수성(LogP): {'유지' if logp_ok else '변화'} ({metrics['delta_logp']:+.3f})")

    sa_ok = metrics["delta_sa_score"] < 0.5
    details.append(f"합성용이성(SA): {'유지/개선' if sa_ok else '악화'} ({metrics['delta_sa_score']:+.3f})")

    checks = [c for c in [shape_ok, qed_ok, logp_ok, sa_ok] if c is not None]
    ok_ratio = sum(checks) / len(checks)

    if ok_ratio >= 0.75:
        verdict = "활성/물성 보존 양호 — 자동 승인 가능"
    elif ok_ratio >= 0.5:
        verdict = "일부 지표 변화 — 검토 권장"
    else:
        verdict = "다수 지표 변화 — 사람 검토 필요"

    return {"verdict": verdict, "details": details, "ok_ratio": ok_ratio}

In [14]:
print("=== BCP 케이스 ===")
result_bcp = classify_activity_risk_v2(metrics_bcp)
for d in result_bcp["details"]:
    print(f"  {d}")
print(f"종합: {result_bcp['verdict']} (보존비율 {result_bcp['ok_ratio']*100:.0f}%)")

print("\n=== 무수물 케이스 ===")
result_anh = classify_activity_risk_v2(metrics_anh)
for d in result_anh["details"]:
    print(f"  {d}")
print(f"종합: {result_anh['verdict']} (보존비율 {result_anh['ok_ratio']*100:.0f}%)")

=== BCP 케이스 ===
  2D 연결성: 상이 (Tanimoto 0.489)
  3D 형태: 보존 (회전반경 -7.2%)
  약물유사성(QED): 유지 (+0.001)
  소수성(LogP): 변화 (-1.008)
  합성용이성(SA): 악화 (+1.879)
종합: 일부 지표 변화 — 검토 권장 (보존비율 50%)

=== 무수물 케이스 ===
  2D 연결성: 상이 (Tanimoto 0.417)
  3D 형태: 변화 (회전반경 -31.6%)
  약물유사성(QED): 변화 (+0.108)
  소수성(LogP): 유지 (-0.005)
  합성용이성(SA): 유지/개선 (-0.801)
종합: 일부 지표 변화 — 검토 권장 (보존비율 50%)


In [15]:
def classify_activity_risk_v3(metrics):
    """3D 형태 보존 여부를 1차 판단축으로 삼고(표적 결합 형태 유지가
    가장 직접적인 활성 근사 지표), QED/LogP/SA는 보조 경고로 덧붙인다."""
    if metrics is None:
        return {"verdict": "판정 불가", "details": []}

    details = []
    warnings = []

    conn_ok = metrics["tanimoto"] >= 0.5
    details.append(f"2D 연결성: {'유사' if conn_ok else '상이'} (Tanimoto {metrics['tanimoto']:.3f})")

    shape_ok = None
    if metrics["shape_available"] and metrics.get("delta_rog_pct") is not None:
        shape_ok = abs(metrics["delta_rog_pct"]) < 15
        details.append(f"3D 형태: {'보존' if shape_ok else '변화'} (회전반경 {metrics['delta_rog_pct']:+.1f}%)")
    else:
        details.append("3D 형태: 계산 불가")

    qed_ok = abs(metrics["delta_qed"]) < 0.1
    details.append(f"약물유사성(QED): {'유지' if qed_ok else '변화'} ({metrics['delta_qed']:+.3f})")
    if not qed_ok:
        warnings.append("QED 변화")

    logp_ok = abs(metrics["delta_logp"]) < 1.0
    details.append(f"소수성(LogP): {'유지' if logp_ok else '변화'} ({metrics['delta_logp']:+.3f})")
    if not logp_ok:
        warnings.append("LogP 변화")

    sa_ok = metrics["delta_sa_score"] < 0.5
    details.append(f"합성용이성(SA): {'유지/개선' if sa_ok else '악화'} ({metrics['delta_sa_score']:+.3f})")
    if not sa_ok:
        warnings.append("합성난이도 증가")

    # 1차 분기: 3D 형태 보존 여부가 핵심
    if shape_ok is None:
        verdict = "3D 형태 계산 불가 — 2D 지표만으로 판단, 신뢰도 낮음"
    elif shape_ok:
        if conn_ok:
            verdict = "구조·형태 모두 보존 — 활성 유지 가능성 높음"
        else:
            verdict = "2D 연결성은 크게 바뀌었으나 3D 형태는 보존됨 (bioisostere 가능성) — 활성 유지 기대"
    else:
        verdict = "3D 형태 자체가 크게 변화 — 표적 결합 형태 훼손 우려, 사람 검토 필요"

    if warnings:
        verdict += f" [보조 경고: {', '.join(warnings)}]"

    return {"verdict": verdict, "details": details, "shape_ok": shape_ok, "warnings": warnings}

In [16]:
print("=== BCP 케이스 ===")
result_bcp_v3 = classify_activity_risk_v3(metrics_bcp)
for d in result_bcp_v3["details"]:
    print(f"  {d}")
print(f"종합: {result_bcp_v3['verdict']}")

print("\n=== 무수물 케이스 ===")
result_anh_v3 = classify_activity_risk_v3(metrics_anh)
for d in result_anh_v3["details"]:
    print(f"  {d}")
print(f"종합: {result_anh_v3['verdict']}")

=== BCP 케이스 ===
  2D 연결성: 상이 (Tanimoto 0.489)
  3D 형태: 보존 (회전반경 -7.2%)
  약물유사성(QED): 유지 (+0.001)
  소수성(LogP): 변화 (-1.008)
  합성용이성(SA): 악화 (+1.879)
종합: 2D 연결성은 크게 바뀌었으나 3D 형태는 보존됨 (bioisostere 가능성) — 활성 유지 기대 [보조 경고: LogP 변화, 합성난이도 증가]

=== 무수물 케이스 ===
  2D 연결성: 상이 (Tanimoto 0.417)
  3D 형태: 변화 (회전반경 -31.6%)
  약물유사성(QED): 변화 (+0.108)
  소수성(LogP): 유지 (-0.005)
  합성용이성(SA): 유지/개선 (-0.801)
종합: 3D 형태 자체가 크게 변화 — 표적 결합 형태 훼손 우려, 사람 검토 필요 [보조 경고: QED 변화]
